


# **ALL Codes**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install spectral
import tensorflow as tf
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
import gc, torch
gc.collect()
tf.keras.backend.clear_session()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 7.6 MB/s eta 0:00:00


**ULITE MODEL (2024) -- COUNTWISE AND %TAGE WISE TRAINING SAMPLES**

In [3]:

#####################     FULL ULITE MODEL with countwise and percentage wise   ###########################

# Full end-to-end script: training, evaluation, saving maps & reports, zipping outputs.
# Option B: include border pixels by padding; exact samples_per_class training.
import os, time, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral

# ---------------- Config ----------------
dataset = 'IP'   # 'IP','SA','PU','Ho','Bo','KSC'
base = "/content/drive/MyDrive/Colab Notebooks/dataset"   # <-- change to your path
windowSize = 25
#samples_per_class = 15
train_ratio = 0.05   # <-- 10% per class
batch_size = 256
epochs = 100
###########################################################################
#results_folder = f"ulite_results_{dataset}_spc{samples_per_class}"
results_folder = f"ulite_results_{dataset}_per{5}"
os.makedirs(results_folder, exist_ok=True)

# Allow mixed precision if desired (optional)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass
tf.keras.backend.clear_session(); gc.collect()

K_default = 30 if dataset == 'IP' else 15
K = K_default

# ---------------- Data loaders ----------------
def loadData(name):
    if name == 'IP':
        data = sio.loadmat(os.path.join(base, 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(base, 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not supported")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')
'''
# ---------------- Patch generator ----------------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        """
        coords: array of (r,c) in ORIGINAL image coordinates (0..H-1 / 0..W-1)
        The generator pads full_cube internally by half=patch_size//2 and extracts:
            start = r,   slice padded[start : start+patch_size]
        which works because padded has top-left padding of 'half' rows/cols.
        """
        self.coords = np.array(coords, dtype=np.int32)
        self.labels = np.array(labels, dtype=np.int32)
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(self.labels))
        # pad once (so padded index i corresponds to original index i-half)
        self.padded = padWithZeros(full_cube, self.half)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            # In padded array the top-left of a patch centered at (r,c) original is at index r
            # because padded has half rows/cols at top/left.
            r0 = r
            c0 = c
            patch = self.padded[r0:r0+self.patch_size, c0:c0+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y'''

# ---------- Memory-safe Patch Generator ----------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        self.coords = coords
        self.labels = labels
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(labels))
        self.padded = padWithZeros(full_cube, self.half)  # pad once
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# %%%%%%%%------ Fixed Split function (samples_per_class) ------%%%%%%%%%%%%%%%%%----------
'''
def splitTrainTestSet(coords, labels, samples_per_class, random_state=42):
    """
    coords, labels are arrays with same length. labels in [0..n-1]
    Returns coords_train, coords_test, labels_train, labels_test
    Training picks up to samples_per_class from each class (if available).
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = min(samples_per_class, len(idx))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]

'''
    ######### Train Ratio fixed ###################
def splitTrainTestSet_ratio(coords, labels, train_ratio=0.05, random_state=42):
    """
    coords, labels arrays with same length.
    Takes train_ratio fraction from each class for training.
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]


# ---------------- ULite-R2HCN blocks (simplified/robust) ----------------
from tensorflow.keras import layers
'''
def R2SpectralBlock(x):
    in_ch = int(x.shape[-1])
    y = layers.Conv3D(filters=max(8, in_ch), kernel_size=(1,1,1), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    y = layers.Conv3D(filters=max(8, in_ch//2 + 1), kernel_size=(1,1,1), padding='same', activation='relu')(y)
    y = layers.BatchNormalization()(y)
    return y

def R2SpatialBlock(x):
    y = layers.Conv3D(filters=max(16, int(x.shape[-1])), kernel_size=(3,3,3), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    # collapse spectral & channel dims
    h = y.shape[1]; w = y.shape[2]; d = y.shape[3]; ch = y.shape[4]
    y_resh = layers.Reshape((h, w, d*ch))(y)
    y_resh = layers.SeparableConv2D(filters=max(16, ch*2), kernel_size=(3,3), padding='same', activation='relu')(y_resh)
    y_resh = layers.BatchNormalization()(y_resh)
    y_out = layers.Reshape((h, w, 1, int(y_resh.shape[-1])))(y_resh)
    return y_out

def build_ulite_r2hcn(windowSize, K, num_classes):
    inp = layers.Input(shape=(windowSize, windowSize, K, 1), dtype='float32')
    x = layers.Conv3D(filters=8, kernel_size=(3,3,7), padding='valid', activation='relu')(inp)
    x = layers.Conv3D(filters=16, kernel_size=(3,3,5), padding='valid', activation='relu')(x)
    x = R2SpectralBlock(x)
    x = R2SpatialBlock(x)
    # flatten 3D to 2D conv input
    shape1 = x.shape[1]; shape2 = x.shape[2]; shape3 = x.shape[3]; shape4 = x.shape[4]
    x = layers.Reshape((shape1, shape2, int(shape3*shape4)))(x)
    x = layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(pool_size=(2,2))(x)
    x = layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model
'''
def R2SpectralBlock(x, groups=4):
    """
    Lightweight spectral block: grouped 1x1 spectral mixing followed by small conv.
    groups: number of groups for group conv to reduce redundancy.
    """
    # x shape: (H, W, Bands, C) or (S,S,K,channels) - we operate on last axis channels
    # Use pointwise grouped conv across spectral axis: implement as Conv2D with kernel (1,1)
    # but we want to mix spectral bands — since input patch shape is (S,S,K,1),
    # we'll squeeze channels dimension and use Conv3D along spectral dimension if needed.
    # Simpler: use a 1x1x1 Conv3D grouped via channel split to simulate grouped spectral mixing.
    in_channels = x.shape[-1]
    # pointwise conv (1x1) along spatial dims and 1x1 spectral mixing via Conv3D with kernel (1,1,1)
    # We'll simulate grouping by splitting channels and applying small Dense-like mixing
    # Implementation chosen for compactness:
    y = layers.Conv3D(filters=max(4, in_channels), kernel_size=(1,1,1), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    # 1x1 spectral projection (reduce channels)
    y = layers.Conv3D(filters=max(4, in_channels//2 + 1), kernel_size=(1,1,1), padding='same', activation='relu')(y)
    y = layers.BatchNormalization()(y)
    return y

def R2SpatialBlock(x):
    """
    Lightweight spatial block: depthwise-separable-like 3D conv with small kernels to capture local spatial context.
    """
    # Apply small 3D conv followed by a separable 2D conv
    y = layers.Conv3D(filters= max(8, int(x.shape[-1]) ), kernel_size=(3,3,3), padding='same', activation='relu')(x)
    y = layers.BatchNormalization()(y)
    # Squeeze spectral dimension by a 1x1 conv then apply 2D separable conv
    # reshape from (S,S,K,C) -> (S,S,K*C,1) not necessary; we apply Conv2D on last two dims after reshape
    shape = y.shape
    # collapse spectral and channel dims for Conv2D processing
    # Use Keras operations for getting shape components
    bsize = tf.keras.ops.shape(y)[0]
    h = tf.keras.ops.shape(y)[1]
    w = tf.keras.ops.shape(y)[2]
    d = tf.keras.ops.shape(y)[3]
    ch = tf.keras.ops.shape(y)[4]
    # merge spectral & channel dims
    y_resh = layers.Reshape((h, w, d*ch))(y)
    y_resh = layers.SeparableConv2D(filters=max(8, ch*2), kernel_size=(3,3), padding='same', activation='relu')(y_resh)
    y_resh = layers.BatchNormalization()(y_resh)
    # project back to a compact 3D-like shape by adding a spectral pseudo-dim (1)
    y_out = layers.Reshape((h, w, 1, int(y_resh.shape[-1]//1)))(y_resh)
    return y_out

# ---------- Build ULite-R2HCN model ----------
def build_ulite_r2hcn(windowSize, K, num_classes):
    # Input shape: (S,S,K,1)
    inp = layers.Input(shape=(windowSize, windowSize, K, 1), dtype='float32')
    # initial spectral-reduction conv (light)
    x = layers.Conv3D(filters=8, kernel_size=(3,3,7), padding='valid', activation='relu')(inp)   # small spectral kernel
    x = layers.Conv3D(filters=16, kernel_size=(3,3,5), padding='valid', activation='relu')(x)
    # apply R2SpectralBlock
    x = R2SpectralBlock(x)
    # R2SpatialBlock
    x = R2SpatialBlock(x)
    # further light convs
    x = layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(layers.Reshape((x.shape[1], x.shape[2], x.shape[3]*x.shape[4]))(x))
    x = layers.MaxPooling2D(pool_size=(2,2))(x)
    x = layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    # small classifier head
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)  # cast to float32 for numeric stability
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model
# ---------------- Pipeline ----------------
print("Loading data...")
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())
print(f"Dataset {dataset}: H={X_full.shape[0]}, W={X_full.shape[1]}, Bands={X_full.shape[2]}, Classes={n_classes}")
total_labeled = int(np.sum(y_full > 0))
print("Total labeled pixels in full map:", total_labeled)

print("Applying PCA ...")
X_pca, pca = applyPCA(X_full, numComponents=K)

# collect coords & labels for ALL labeled pixels (include edges)
coords, labels = [], []
H_img, W_img = X_pca.shape[0], X_pca.shape[1]
for r in range(0, H_img):
    for c in range(0, W_img):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))          # original coordinates (centers)
            labels.append(lab - 1)         # zero-based labels
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)
print("Collected coords (should equal total labeled):", len(labels))

#%%%%%%%%%% split using exact samples_per_class per class    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
'''train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet(coords, labels, samples_per_class)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx), "Total:", len(ytrain_idx)+len(ytest_idx))
'''
# split using 10% per class
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet_ratio(coords, labels, train_ratio)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx))


# create generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# build model
print("Building model...")
model = build_ulite_r2hcn(windowSize, K, n_classes)
model.summary()
params = model.count_params()
print(f"Total parameters: {params:,}")

# save model summary
with open(os.path.join(results_folder, 'ulite_model_summary.txt'), 'w') as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))

# Training
print("Training...")
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic
print(f"Training took {train_time:.2f} s")

# Save training curve
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Training curves')
plt.savefig(os.path.join(results_folder, 'ulite_training_curves.png'), dpi=150)
plt.close()

# Evaluate on test set
print("Evaluating on test set...")
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx

classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# confusion figure
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='viridis')
plt.colorbar()
plt.title('Confusion Matrix (ULite replica)')
plt.savefig(os.path.join(results_folder, 'ulite_confusion.png'), dpi=150)
plt.close()

# Save results text
with open(os.path.join(results_folder, 'ulite_results.txt'), 'w') as f:
    f.write("ULite-R2HCN (replica) results\n")
    f.write(f"Dataset: {dataset}\n")
    f.write(f"PCA components: {K}, Window size: {windowSize}\n")
    f.write(f"Params: {params}\n")
    f.write(f"Train time (s): {train_time:.2f}\n")
    f.write(f"Test inference time (s total): {test_time:.4f}\n")
    f.write(f"Overall Accuracy (OA): {oa*100:.2f}%\n")
    f.write(f"Average Accuracy (AA): {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")
    f.write("Classwise accuracies (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")
    f.write("Classification report:\n")
    f.write(classification + "\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm) + "\n")

# Predict full map (batched) for visualization -- include all labeled pixels
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H_img, W_img = y_full.shape
outputs = np.zeros((H_img, W_img), dtype=np.int32)  # will store class labels in 1..n_classes, 0 for background

coords_all = [(r, c) for r in range(0, H_img) for c in range(0, W_img) if y_full[r, c] > 0]
B = 2048
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r:r+PATCH, c:c+PATCH, :]   # padded Xp: indexed by r..r+PATCH
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = int(p) + 1  # keep 1-based label for visualization

# Save maps using spectral (background remains 0 -> black)
spectral.save_rgb(os.path.join(results_folder, f'ulite_classified_map_{dataset}.jpg'), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f'ulite_ground_truth_{dataset}.jpg'), y_full.astype(int), colors=spectral.spy_colors)

# Save model weights
model.save_weights(os.path.join(results_folder, 'ulite_weights.weights.h5'))

# Zip outputs
#zip_path = os.path.join('.', f'ulite_outputs_{dataset}_spc{samples_per_class}.zip')
zip_path = os.path.join('.', f'ulite_outputs_{dataset}_per{5}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print("ULite replica pipeline done.")
print("Saved results in folder:", results_folder)
print("Zipped outputs:", zip_path)

Loading data...
Dataset IP: H=145, W=145, Bands=200, Classes=16
Total labeled pixels in full map: 10249
Applying PCA ...
Collected coords (should equal total labeled): 10249
Train samples: 505 Test samples: 9744
Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 25, 25, 30, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 23, 23, 24, 8)  │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 21, 21, 20, 16) │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 21, 21, 20, 16) │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 21, 21, 20, 16) │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 21, 21, 20, 9)  │           153 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 20, 9)  │            36 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_4 (Conv3D)               │ (None, 21, 21, 20, 9)  │         2,196 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 21, 21, 20, 9)  │            36 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 21, 21, 180)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 21, 21, 18)     │         4,878 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 21, 21, 18)     │            72 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 21, 21, 1, 18)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 21, 21, 18)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 21, 21, 32)     │         5,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 10, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,907 (167.61 KB)

 Trainable params: 42,803 (167.20 KB)

...

Training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2/2 - 52s - 26s/step - accuracy: 0.1248 - loss: 2.7511 - val_accuracy: 0.2394 - val_loss: 2.7674
Epoch 2/100
2/2 - 3s - 2s/step - accuracy: 0.2158 - loss: 2.5381 - val_accuracy: 0.2394 - val_loss: 2.7634
Epoch 3/100
2/2 - 3s - 2s/step - accuracy: 0.2356 - loss: 2.4099 - val_accuracy: 0.3357 - val_loss: 2.7589
Epoch 4/100
2/2 - 3s - 2s/step - accuracy: 0.2931 - loss: 2.2754 - val_accuracy: 0.1369 - val_loss: 2.7545
Epoch 5/100
2/2 - 3s - 2s/step - accuracy: 0.3366 - loss: 2.1058 - val_accuracy: 0.1297 - val_loss: 2.7507
Epoch 6/100
2/2 - 3s - 2s/step - accuracy: 0.3386 - loss: 1.9905 - val_accuracy: 0.1322 - val_loss: 2.7480
Epoch 7/100
2/2 - 3s - 2s/step - accuracy: 0.3743 - loss: 1.8914 - val_accuracy: 0.2457 - val_loss: 2.7463
Epoch 8/100
2/2 - 3s - 2s/step - accuracy: 0.4119 - loss: 1.7981 - val_accuracy: 0.2424 - val_loss: 2.7450
Epoch 9/100
2/2 - 3s - 2s/step - accuracy: 0.4455 - loss: 1.7089 - val_accuracy: 0.2398 - val_loss: 2.7427
Epoch 10/100
2/2 - 6s - 3s/step - accuracy: 0.4

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Predicting full map (batched)...
ULite replica pipeline done.
Saved results in folder: ulite_results_IP_per5
Zipped outputs: ./ulite_outputs_IP_per5.zip


**PROPOSED LIGHTWEIGHT (LhSSN) MODEL -- COUNTWISE AND % TAGE WISE TRAINIING**

In [8]:
############  Proposed  LightModel with countwise and percentage wise (Performed changes in ULITE MOdEL) #########################

# Full end-to-end script: training, evaluation, saving maps & reports, zipping outputs.
# Option B: include border pixels by padding; exact samples_per_class training.
import os, time, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral

# ---------------- Config ----------------
dataset = 'KSC'   # 'IP','SA','PU','Ho','Bo','KSC'
base = "/content/drive/MyDrive/Colab Notebooks/dataset"   # <-- change to your path
windowSize = 25
#samples_per_class = 15
train_ratio = 0.05   # <-- 10% per class
batch_size = 256
epochs = 100
###########################################################################
#results_folder = f"LhSSN_results_{dataset}_spc{samples_per_class}"
results_folder = f"LhSSN_results_{dataset}_per{5}"
os.makedirs(results_folder, exist_ok=True)

# Allow mixed precision if desired (optional)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass
tf.keras.backend.clear_session(); gc.collect()

K_default = 30 if dataset == 'IP' else 15
K = K_default

# ---------------- Data loaders ----------------
def loadData(name):
    if name == 'IP':
        data = sio.loadmat(os.path.join(base, 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(base, 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not supported")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')
'''
# ---------------- Patch generator ----------------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        """
        coords: array of (r,c) in ORIGINAL image coordinates (0..H-1 / 0..W-1)
        The generator pads full_cube internally by half=patch_size//2 and extracts:
            start = r,   slice padded[start : start+patch_size]
        which works because padded has top-left padding of 'half' rows/cols.
        """
        self.coords = np.array(coords, dtype=np.int32)
        self.labels = np.array(labels, dtype=np.int32)
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(self.labels))
        # pad once (so padded index i corresponds to original index i-half)
        self.padded = padWithZeros(full_cube, self.half)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            # In padded array the top-left of a patch centered at (r,c) original is at index r
            # because padded has half rows/cols at top/left.
            r0 = r
            c0 = c
            patch = self.padded[r0:r0+self.patch_size, c0:c0+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y'''

# ---------- Memory-safe Patch Generator ----------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        self.coords = coords
        self.labels = labels
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(labels))
        self.padded = padWithZeros(full_cube, self.half)  # pad once
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# ---------------- Fixed Split function (samples_per_class) ----------------
'''def splitTrainTestSet(coords, labels, samples_per_class, random_state=42):
    """
    coords, labels are arrays with same length. labels in [0..n-1]
    Returns coords_train, coords_test, labels_train, labels_test
    Training picks up to samples_per_class from each class (if available).
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = min(samples_per_class, len(idx))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]
'''

    ######### Train Ratio fixed ###################
def splitTrainTestSet_ratio(coords, labels, train_ratio=0.05, random_state=42):
    """
    coords, labels arrays with same length.
    Takes train_ratio fraction from each class for training.
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]


# ---------------- LhSSN blocks (simplified/robust) ----------------
from tensorflow.keras import layers

########################################################################
# --------- Build Model (EXACT architecture you used) ---------
def build_model(S, L, num_classes):
    inp = tf.keras.layers.Input((S, S, L, 1))
    x = tf.keras.layers.Conv3D(filters=8,  kernel_size=(3,3,7), activation='relu')(inp)
    x = tf.keras.layers.Conv3D(filters=16, kernel_size=(3,3,5), activation='relu')(x)
    x = tf.keras.layers.Conv3D(filters=32, kernel_size=(3,3,3), activation='relu')(x)
    # reshape to 2D convs
    conv3d_shape = x.shape  # (None, H, W, D, C)
    x = tf.keras.layers.Reshape((conv3d_shape[1], conv3d_shape[2],
                                 conv3d_shape[3]*conv3d_shape[4]))(x)
    x = tf.keras.layers.Conv2D(filters=24,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.Conv2D(filters=96,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2,2))(x)
    x = tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    # Adam 0.001 as in your code
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

    ########################################################################################
# ---------------- Pipeline ----------------
print("Loading data...")
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())
print(f"Dataset {dataset}: H={X_full.shape[0]}, W={X_full.shape[1]}, Bands={X_full.shape[2]}, Classes={n_classes}")
total_labeled = int(np.sum(y_full > 0))
print("Total labeled pixels in full map:", total_labeled)

print("Applying PCA ...")
X_pca, pca = applyPCA(X_full, numComponents=K)

# collect coords & labels for ALL labeled pixels (include edges)
coords, labels = [], []
H_img, W_img = X_pca.shape[0], X_pca.shape[1]
for r in range(0, H_img):
    for c in range(0, W_img):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))          # original coordinates (centers)
            labels.append(lab - 1)         # zero-based labels
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)
print("Collected coords (should equal total labeled):", len(labels))

#%%%%%%%%%%%%%%%%%%%%%%%%% split using exact samples_per_class per class   %%%%%%%%%%%%%
'''train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet(coords, labels, samples_per_class)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx), "Total:", len(ytrain_idx)+len(ytest_idx))
'''
# split using 10% per class
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet_ratio(coords, labels, train_ratio)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx))


# create generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# build model
print("Building model...")
model = build_model(windowSize, K, n_classes)
model.summary()
params = model.count_params()
print(f"Total parameters: {params:,}")

# save model summary
with open(os.path.join(results_folder, 'LhSSN_model_summary.txt'), 'w') as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))

# Training
print("Training...")
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic
print(f"Training took {train_time:.2f} s")

# Save training curve
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Training curves')
plt.savefig(os.path.join(results_folder, 'LhSSN_training_curves.png'), dpi=150)
plt.close()

# Evaluate on test set
print("Evaluating on test set...")
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx

classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# confusion figure
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='viridis')
plt.colorbar()
plt.title('Confusion Matrix (LhSSN replica)')
plt.savefig(os.path.join(results_folder, 'LhSSN_confusion.png'), dpi=150)
plt.close()

# Save results text
with open(os.path.join(results_folder, 'LhSSN_results.txt'), 'w') as f:
    f.write("LhSSN (replica) results\n")
    f.write(f"Dataset: {dataset}\n")
    f.write(f"PCA components: {K}, Window size: {windowSize}\n")
    f.write(f"Params: {params}\n")
    f.write(f"Train time (s): {train_time:.2f}\n")
    f.write(f"Test inference time (s total): {test_time:.4f}\n")
    f.write(f"Overall Accuracy (OA): {oa*100:.2f}%\n")
    f.write(f"Average Accuracy (AA): {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")
    f.write("Classwise accuracies (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")
    f.write("Classification report:\n")
    f.write(classification + "\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm) + "\n")

# Predict full map (batched) for visualization -- include all labeled pixels
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H_img, W_img = y_full.shape
outputs = np.zeros((H_img, W_img), dtype=np.int32)  # will store class labels in 1..n_classes, 0 for background

coords_all = [(r, c) for r in range(0, H_img) for c in range(0, W_img) if y_full[r, c] > 0]
B = 2048
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r:r+PATCH, c:c+PATCH, :]   # padded Xp: indexed by r..r+PATCH
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = int(p) + 1  # keep 1-based label for visualization

# Save maps using spectral (background remains 0 -> black)
spectral.save_rgb(os.path.join(results_folder, f'LhSSN_classified_map_{dataset}.jpg'), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f'LhSSN_ground_truth_{dataset}.jpg'), y_full.astype(int), colors=spectral.spy_colors)

# Save model weights
model.save_weights(os.path.join(results_folder, 'LhSSN_weights.weights.h5'))

# Zip outputs
#zip_path = os.path.join('.', f'LhSSN_outputs_{dataset}_spc{samples_per_class}.zip')
zip_path = os.path.join('.', f'LhSSN_outputs_{dataset}_per{5}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print("LhSSN replica pipeline done.")
print("Saved results in folder:", results_folder)
print("Zipped outputs:", zip_path)

Loading data...
Dataset KSC: H=512, W=614, Bands=176, Classes=13
Total labeled pixels in full map: 5211
Applying PCA ...
Collected coords (should equal total labeled): 5211
Train samples: 256 Test samples: 4955
Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 25, 25, 15, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 23, 23, 9, 8)   │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 21, 21, 5, 16)  │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 19, 19, 3, 32)  │        13,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 19, 19, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 17, 17, 24)     │        20,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 15, 15, 96)     │        20,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 7, 7, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 128)      │       110,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 13)             │           845 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 198,069 (773.71 KB)

 Trainable params: 198,069 (773.71 KB)

...

Training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 - 9s - 9s/step - accuracy: 0.1016 - loss: 2.5619 - val_accuracy: 0.1887 - val_loss: 2.5434
Epoch 2/100
1/1 - 1s - 645ms/step - accuracy: 0.1211 - loss: 2.5437 - val_accuracy: 0.1617 - val_loss: 2.5036
Epoch 3/100
1/1 - 1s - 663ms/step - accuracy: 0.1602 - loss: 2.5068 - val_accuracy: 0.0965 - val_loss: 2.4376
Epoch 4/100
1/1 - 1s - 668ms/step - accuracy: 0.1602 - loss: 2.4428 - val_accuracy: 0.0965 - val_loss: 2.4179
Epoch 5/100
1/1 - 1s - 627ms/step - accuracy: 0.1406 - loss: 2.4871 - val_accuracy: 0.0965 - val_loss: 2.3929
Epoch 6/100
1/1 - 1s - 638ms/step - accuracy: 0.1094 - loss: 2.4115 - val_accuracy: 0.0965 - val_loss: 2.3822
Epoch 7/100
1/1 - 1s - 647ms/step - accuracy: 0.1172 - loss: 2.3873 - val_accuracy: 0.1449 - val_loss: 2.3777
Epoch 8/100
1/1 - 1s - 754ms/step - accuracy: 0.1328 - loss: 2.3917 - val_accuracy: 0.1633 - val_loss: 2.3677
Epoch 9/100
1/1 - 1s - 728ms/step - accuracy: 0.2070 - loss: 2.3668 - val_accuracy: 0.1697 - val_loss: 2.3467
Epoch 10/100
1/1 - 1s - 1

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Predicting full map (batched)...
LhSSN replica pipeline done.
Saved results in folder: LhSSN_results_KSC_per5
Zipped outputs: ./LhSSN_outputs_KSC_per5.zip


**HybridSN (Original Model) Count and Percentage Wise**

In [5]:
############  HYBRIDSN with countwise and percentage wise (Performed changes in ULITE MOdEL) #########################

# Full end-to-end script: training, evaluation, saving maps & reports, zipping outputs.
# Option B: include border pixels by padding; exact samples_per_class training.
import os, time, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral

# ---------------- Config ----------------
dataset = 'Bo'   # 'IP','SA','PU','Ho','Bo','KSC'
base = "/content/drive/MyDrive/Colab Notebooks/dataset"   # <-- change to your path
windowSize = 25
#samples_per_class = 15
train_ratio = 0.05   # <-- 10% per class
batch_size = 256
epochs = 100
#################  CHANGE BELOW  LINE             #########################################
#results_folder = f"HybridSN_results_{dataset}_spc{samples_per_class}"
results_folder = f"HybridSN_results_{dataset}_per{5}"
###############################################################
os.makedirs(results_folder, exist_ok=True)

# Allow mixed precision if desired (optional)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass
tf.keras.backend.clear_session(); gc.collect()

K_default = 30 if dataset == 'IP' else 15
K = K_default

# ---------------- Data loaders ----------------
def loadData(name):
    if name == 'IP':
        data = sio.loadmat(os.path.join(base, 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(base, 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not supported")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')
'''
# ---------------- Patch generator ----------------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        """
        coords: array of (r,c) in ORIGINAL image coordinates (0..H-1 / 0..W-1)
        The generator pads full_cube internally by half=patch_size//2 and extracts:
            start = r,   slice padded[start : start+patch_size]
        which works because padded has top-left padding of 'half' rows/cols.
        """
        self.coords = np.array(coords, dtype=np.int32)
        self.labels = np.array(labels, dtype=np.int32)
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(self.labels))
        # pad once (so padded index i corresponds to original index i-half)
        self.padded = padWithZeros(full_cube, self.half)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            # In padded array the top-left of a patch centered at (r,c) original is at index r
            # because padded has half rows/cols at top/left.
            r0 = r
            c0 = c
            patch = self.padded[r0:r0+self.patch_size, c0:c0+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y'''

# ---------- Memory-safe Patch Generator ----------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        self.coords = coords
        self.labels = labels
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(labels))
        self.padded = padWithZeros(full_cube, self.half)  # pad once
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# %%%%%%%%%%%%%---- Fixed Split function (samples_per_class) --%%%%%%%%%%%%%-------
'''def splitTrainTestSet(coords, labels, samples_per_class, random_state=42):
    """
    coords, labels are arrays with same length. labels in [0..n-1]
    Returns coords_train, coords_test, labels_train, labels_test
    Training picks up to samples_per_class from each class (if available).
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = min(samples_per_class, len(idx))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]'''


    ######### Train Ratio fixed ###################
def splitTrainTestSet_ratio(coords, labels, train_ratio=0.05, random_state=42):
    """
    coords, labels arrays with same length.
    Takes train_ratio fraction from each class for training.
    """
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]


# ---------------- HybridSN blocks (simplified/robust) ----------------
from tensorflow.keras import layers

########################################################################
# --------- Build Model (EXACT architecture you used) ---------
def build_model(S, L, num_classes):
    inp = tf.keras.layers.Input((S, S, L, 1))
    x = tf.keras.layers.Conv3D(filters=8,  kernel_size=(3,3,7), activation='relu')(inp)
    x = tf.keras.layers.Conv3D(filters=16, kernel_size=(3,3,5), activation='relu')(x)
    x = tf.keras.layers.Conv3D(filters=32, kernel_size=(3,3,3), activation='relu')(x)
    # reshape to 2D convs
    conv3d_shape = x.shape  # (None, H, W, D, C)
    x = tf.keras.layers.Reshape((conv3d_shape[1], conv3d_shape[2],
                                 conv3d_shape[3]*conv3d_shape[4]))(x)
    x = tf.keras.layers.Conv2D(filters=64,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.Flatten()(x)
    #x = tf.keras.layers.Conv2D(filters=96,  kernel_size=(3,3), activation='relu')(x)
    #x = tf.keras.layers.MaxPooling2D(pool_size=(2,2))(x)
    #x = tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu')(x)
    #x = tf.keras.layers.GlobalAveragePooling2D()(x)
    #x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dense(units=256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    # Adam 0.001 as in your code
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

    ########################################################################################
# ---------------- Pipeline ----------------
print("Loading data...")
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())
print(f"Dataset {dataset}: H={X_full.shape[0]}, W={X_full.shape[1]}, Bands={X_full.shape[2]}, Classes={n_classes}")
total_labeled = int(np.sum(y_full > 0))
print("Total labeled pixels in full map:", total_labeled)

print("Applying PCA ...")
X_pca, pca = applyPCA(X_full, numComponents=K)

# collect coords & labels for ALL labeled pixels (include edges)
coords, labels = [], []
H_img, W_img = X_pca.shape[0], X_pca.shape[1]
for r in range(0, H_img):
    for c in range(0, W_img):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))          # original coordinates (centers)
            labels.append(lab - 1)         # zero-based labels
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)
print("Collected coords (should equal total labeled):", len(labels))

#%%%%%%%%%%%%%%%%%%% split using exact samples_per_class per class   %%%%%%%%%%%%%%%%%%%%%%%
'''train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet(coords, labels, samples_per_class)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx), "Total:", len(ytrain_idx)+len(ytest_idx))
'''
# split using 10% per class
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet_ratio(coords, labels, train_ratio)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx))


# create generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# build model
print("Building model...")
model = build_model(windowSize, K, n_classes)
model.summary()
params = model.count_params()
print(f"Total parameters: {params:,}")

# save model summary
with open(os.path.join(results_folder, 'HybridSN_model_summary.txt'), 'w') as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))

# Training
print("Training...")
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic
print(f"Training took {train_time:.2f} s")

# Save training curve
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Training curves')
plt.savefig(os.path.join(results_folder, 'HybridSN_training_curves.png'), dpi=150)
plt.close()

# Evaluate on test set
print("Evaluating on test set...")
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx

classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# confusion figure
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='viridis')
plt.colorbar()
plt.title('Confusion Matrix (HybridSN replica)')
plt.savefig(os.path.join(results_folder, 'HybridSN_confusion.png'), dpi=150)
plt.close()

# Save results text
with open(os.path.join(results_folder, 'HybridSN_results.txt'), 'w') as f:
    f.write("HybridSN (replica) results\n")
    f.write(f"Dataset: {dataset}\n")
    f.write(f"PCA components: {K}, Window size: {windowSize}\n")
    f.write(f"Params: {params}\n")
    f.write(f"Train time (s): {train_time:.2f}\n")
    f.write(f"Test inference time (s total): {test_time:.4f}\n")
    f.write(f"Overall Accuracy (OA): {oa*100:.2f}%\n")
    f.write(f"Average Accuracy (AA): {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")
    f.write("Classwise accuracies (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")
    f.write("Classification report:\n")
    f.write(classification + "\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm) + "\n")

# Predict full map (batched) for visualization -- include all labeled pixels
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H_img, W_img = y_full.shape
outputs = np.zeros((H_img, W_img), dtype=np.int32)  # will store class labels in 1..n_classes, 0 for background

coords_all = [(r, c) for r in range(0, H_img) for c in range(0, W_img) if y_full[r, c] > 0]
B = 2048
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r:r+PATCH, c:c+PATCH, :]   # padded Xp: indexed by r..r+PATCH
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = int(p) + 1  # keep 1-based label for visualization

# Save maps using spectral (background remains 0 -> black)
spectral.save_rgb(os.path.join(results_folder, f'HybridSN_classified_map_{dataset}.jpg'), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f'HybridSN_ground_truth_{dataset}.jpg'), y_full.astype(int), colors=spectral.spy_colors)

# Save model weights
model.save_weights(os.path.join(results_folder, 'HybridSN_weights.weights.h5'))

# Zip outputs
#zip_path = os.path.join('.', f'HybridSN_outputs_{dataset}_spc{samples_per_class}.zip')
zip_path = os.path.join('.', f'HybridSN_outputs_{dataset}_per{5}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print("HybridSN replica pipeline done.")
print("Saved results in folder:", results_folder)
print("Zipped outputs:", zip_path)

Loading data...
Dataset Bo: H=1476, W=256, Bands=145, Classes=14
Total labeled pixels in full map: 3248
Applying PCA ...
Collected coords (should equal total labeled): 3248
Train samples: 156 Test samples: 3092
Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 25, 25, 15, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 23, 23, 9, 8)   │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 21, 21, 5, 16)  │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 19, 19, 3, 32)  │        13,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 19, 19, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 17, 17, 64)     │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,735,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 14)             │         1,806 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,845,438 (18.48 MB)

 Trainable params: 4,845,438 (18.48 MB)

...

Total parameters: 4,845,438


Training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 - 10s - 10s/step - accuracy: 0.1026 - loss: 2.6339 - val_accuracy: 0.0938 - val_loss: 2.5802
Epoch 2/100
1/1 - 0s - 423ms/step - accuracy: 0.1410 - loss: 2.5875 - val_accuracy: 0.2791 - val_loss: 2.5456
Epoch 3/100
1/1 - 0s - 382ms/step - accuracy: 0.1923 - loss: 2.4854 - val_accuracy: 0.2367 - val_loss: 2.4320
Epoch 4/100
1/1 - 0s - 361ms/step - accuracy: 0.1987 - loss: 2.3979 - val_accuracy: 0.3215 - val_loss: 2.2402
Epoch 5/100
1/1 - 0s - 368ms/step - accuracy: 0.3526 - loss: 2.1895 - val_accuracy: 0.3383 - val_loss: 2.0766
Epoch 6/100
1/1 - 0s - 366ms/step - accuracy: 0.3590 - loss: 2.0278 - val_accuracy: 0.4104 - val_loss: 1.8418
Epoch 7/100
1/1 - 0s - 372ms/step - accuracy: 0.4295 - loss: 1.8306 - val_accuracy: 0.5424 - val_loss: 1.6461
Epoch 8/100
1/1 - 0s - 368ms/step - accuracy: 0.4615 - loss: 1.6694 - val_accuracy: 0.5915 - val_loss: 1.4147
Epoch 9/100
1/1 - 0s - 372ms/step - accuracy: 0.5577 - loss: 1.5214 - val_accuracy: 0.5993 - val_loss: 1.2501
Epoch 10/100
1/1 - 0s -

KeyboardInterrupt: 

**LhSSN Code with Flops**

In [ ]:
# ================================
# LhSSN End-to-end Script (Full) with FLOPs & Params Computation
# ================================
# Make sure required packages are installed in your environment (e.g., Colab)
!pip install spectral keras-flops

import os, time, zipfile, gc, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral

# ---------------- Config ----------------
dataset = 'IP'   # 'IP','SA','PU','Ho','Bo','KSC'
base = "/content/drive/MyDrive/Colab Notebooks/dataset/"   # <-- change to your path
windowSize = 25
samples_per_class = 5
batch_size = 256
epochs = 100

results_folder = f"LhSSN_results_{dataset}_spc{samples_per_class}"
os.makedirs(results_folder, exist_ok=True)

# Optional mixed precision if available
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception:
    pass

tf.keras.backend.clear_session(); gc.collect()

K_default = 30 if dataset == 'IP' else 15
K = K_default

# ---------------- Data loaders ----------------
def loadData(name):
    if name == 'IP':
        data = sio.loadmat(os.path.join(base, 'Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = sio.loadmat(os.path.join(base, 'Indian_pines_gt.mat'))['indian_pines_gt']
    elif name == 'SA':
        data = sio.loadmat(os.path.join(base, 'Salinas_corrected.mat'))['salinas_corrected']
        labels = sio.loadmat(os.path.join(base, 'Salinas_gt.mat'))['salinas_gt']
    elif name == 'Ho':
        data = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    elif name == 'PU':
        data = sio.loadmat(os.path.join(base, 'PaviaU.mat'))['paviaU']
        labels = sio.loadmat(os.path.join(base, 'PaviaU_gt.mat'))['paviaU_gt']
    elif name == 'Bo':
        data = sio.loadmat(os.path.join(base, 'Botswana.mat'))['Botswana']
        labels = sio.loadmat(os.path.join(base, 'Botswana_gt.mat'))['Botswana_gt']
    elif name == 'KSC':
        data = sio.loadmat(os.path.join(base, 'KSC.mat'))['KSC']
        labels = sio.loadmat(os.path.join(base, 'KSC_gt.mat'))['KSC_gt']
    else:
        raise ValueError("Dataset not supported")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')

# ---------------- Patch generator ----------------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=16):
        self.coords = np.array(coords, dtype=np.int32)
        self.labels = np.array(labels, dtype=np.int32)
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(self.labels))
        self.padded = padWithZeros(full_cube, self.half)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# ---------------- Fixed Split function (samples_per_class) ----------------
def splitTrainTestSet(coords, labels, samples_per_class, random_state=42):
    np.random.seed(random_state)
    train_idx, test_idx = [], []
    labels = np.array(labels)
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = min(samples_per_class, len(idx))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    train_idx = np.array(train_idx, dtype=int)
    test_idx = np.array(test_idx, dtype=int)
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]

# ---------------- Model builder (your exact architecture) ----------------
from tensorflow.keras import layers

def build_model(S, L, num_classes):
    inp = tf.keras.layers.Input((S, S, L, 1))
    x = tf.keras.layers.Conv3D(filters=8,  kernel_size=(3,3,7), activation='relu')(inp)
    x = tf.keras.layers.Conv3D(filters=16, kernel_size=(3,3,5), activation='relu')(x)
    x = tf.keras.layers.Conv3D(filters=32, kernel_size=(3,3,3), activation='relu')(x)
    conv3d_shape = x.shape  # (None, H, W, D, C)
    x = tf.keras.layers.Reshape((conv3d_shape[1], conv3d_shape[2],
                                 conv3d_shape[3]*conv3d_shape[4]))(x)
    x = tf.keras.layers.Conv2D(filters=24,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.Conv2D(filters=96,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2,2))(x)
    x = tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

# ---------------- Pipeline ----------------
print("Loading data...")
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())
print(f"Dataset {dataset}: H={X_full.shape[0]}, W={X_full.shape[1]}, Bands={X_full.shape[2]}, Classes={n_classes}")
total_labeled = int(np.sum(y_full > 0))
print("Total labeled pixels in full map:", total_labeled)

print("Applying PCA ...")
X_pca, pca = applyPCA(X_full, numComponents=K)

# collect coords & labels for ALL labeled pixels (center coords)
coords, labels = [], []
H_img, W_img = X_pca.shape[0], X_pca.shape[1]
for r in range(0, H_img):
    for c in range(0, W_img):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))
            labels.append(lab - 1)
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)
print("Collected coords (should equal total labeled):", len(labels))

# split using exact samples_per_class per class
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestSet(coords, labels, samples_per_class)
print("Train samples:", len(ytrain_idx), "Test samples:", len(ytest_idx), "Total:", len(ytrain_idx)+len(ytest_idx))

# create generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# build model
print("Building model...")
model = build_model(windowSize, K, n_classes)
model.summary()
params = model.count_params()
print(f"Total parameters: {params:,}")

# ---------------- Compute FLOPs, params, model size ----------------
# try importing keras_flops, if missing install it (works in Colab)
try:
    from keras_flops import get_flops
except Exception:
    print("keras-flops not found — installing...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "keras-flops"], stdout=subprocess.DEVNULL)
        from keras_flops import get_flops
    except Exception as e:
        print("Failed to install or import keras-flops:", e)
        get_flops = None

# Calculate params (trainable only) and model size
total_params = model.count_params()  # total params (trainable + non-trainable)
trainable_params = int(np.sum([tf.keras.backend.count_params(p) for p in model.trainable_weights]))

# Model size in MB (float32)
bytes_per_param_float32 = 4
model_size_bytes_f32 = total_params * bytes_per_param_float32
model_size_mb_f32 = model_size_bytes_f32 / (1024**2)

# If mixed precision is used, estimate float16 size
bytes_per_param_float16 = 2
model_size_mb_f16 = (total_params * bytes_per_param_float16) / (1024**2)

# Compute FLOPs (batch_size=1). If get_flops unavailable, set to None.
flops = None
if 'get_flops' in globals() and get_flops is not None:
    try:
        flops = get_flops(model, batch_size=1)
    except Exception as e:
        print("keras-flops failed to compute FLOPs:", e)
        flops = None

# Human-readable strings
flops_str = f"{flops:,}" if (flops is not None) else "N/A"
model_size_str = f"{model_size_mb_f32:.6f} MB (float32), ~{model_size_mb_f16:.6f} MB (float16 est)"

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size estimate: {model_size_str}")
print(f"FLOPs (per forward pass, batch=1): {flops_str}")

# Save quick complexity file
with open(os.path.join(results_folder, 'LhSSN_model_flops_params.txt'), 'w') as f:
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Model size (float32): {model_size_mb_f32:.6f} MB\n")
    f.write(f"Model size (float16 est): {model_size_mb_f16:.6f} MB\n")
    f.write(f"FLOPs (batch=1): {flops_str}\n")

# Append complexity info to results file (will exist/ be appended later as well)
results_txt_path = os.path.join(results_folder, 'LhSSN_results.txt')
with open(results_txt_path, 'a') as f:
    f.write("\n=== Model Complexity (auto-computed before training) ===\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Model size (float32): {model_size_mb_f32:.6f} MB\n")
    f.write(f"Model size (float16 est): {model_size_mb_f16:.6f} MB\n")
    f.write(f"FLOPs (per forward pass, batch=1): {flops_str}\n")
    f.write("=======================================\n")

# ---------------- Training ----------------
print("Training...")
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic
print(f"Training took {train_time:.2f} s")

# Save training curve
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('epoch'); plt.legend(); plt.title('Training curves')
plt.savefig(os.path.join(results_folder, 'LhSSN_training_curves.png'), dpi=150)
plt.close()

# ---------------- Evaluation ----------------
print("Evaluating on test set...")
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx

classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# confusion figure
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='viridis')
plt.colorbar()
plt.title('Confusion Matrix (LhSSN replica)')
plt.savefig(os.path.join(results_folder, 'LhSSN_confusion.png'), dpi=150)
plt.close()

# ---------------- Save results text ----------------
with open(os.path.join(results_folder, 'LhSSN_results.txt'), 'a') as f:
    f.write("LhSSN (replica) results\n")
    f.write(f"Dataset: {dataset}\n")
    f.write(f"PCA components: {K}, Window size: {windowSize}\n")
    f.write(f"Samples per class (train): {samples_per_class}\n")
    f.write(f"Params (total): {total_params:,}\n")
    f.write(f"Train time (s): {train_time:.2f}\n")
    f.write(f"Test inference time (s total): {test_time:.4f}\n")
    f.write(f"Overall Accuracy (OA): {oa*100:.2f}%\n")
    f.write(f"Average Accuracy (AA): {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")
    f.write("Classwise accuracies (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")
    f.write("Classification report:\n")
    f.write(classification + "\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm) + "\n")

# ---------------- Full map prediction (batched) ----------------
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H_img, W_img = y_full.shape
outputs = np.zeros((H_img, W_img), dtype=np.int32)

coords_all = [(r, c) for r in range(0, H_img) for c in range(0, W_img) if y_full[r, c] > 0]
B = 2048
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r:r+PATCH, c:c+PATCH, :]
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = int(p) + 1

spectral.save_rgb(os.path.join(results_folder, f'LhSSN_classified_map_{dataset}.jpg'), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f'LhSSN_ground_truth_{dataset}.jpg'), y_full.astype(int), colors=spectral.spy_colors)

# Save model weights
model.save_weights(os.path.join(results_folder, 'LhSSN_weights.weights.h5'))

# Zip outputs
zip_path = os.path.join('.', f'LhSSN_outputs_{dataset}_spc{samples_per_class}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print("LhSSN replica pipeline done.")
print("Saved results in folder:", results_folder)
print("Zipped outputs:", zip_path)


Loading data...
Dataset IP: H=145, W=145, Bands=200, Classes=16
Total labeled pixels in full map: 10249
Applying PCA ...
Collected coords (should equal total labeled): 10249
Train samples: 240 Test samples: 10009 Total: 10249
Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 25, 25, 30, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 23, 23, 24, 8)  │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 21, 21, 20, 16) │         5,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 19, 19, 18, 32) │        13,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 19, 19, 576)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 17, 17, 24)     │       124,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 15, 15, 96)     │        20,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 7, 7, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 128)      │       110,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 301,944 (1.15 MB)

 Trainable params: 301,944 (1.15 MB)

 Non-trainable params: 0 (0.00 B)

Total parameters: 301,944
keras-flops not found — installing...
Failed to install or import keras-flops: "Registering two statistical functions with name 'FusedBatchNormV3,flops'! (Previous registration was in register /usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/registry.py:65)"
Total params: 301,944
Trainable params: 301,944
Model size estimate: 1.151825 MB (float32), ~0.575912 MB (float16 est)
FLOPs (per forward pass, batch=1): N/A
Training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 - 10s - 10s/step - accuracy: 0.0417 - loss: 2.7725 - val_accuracy: 0.0025 - val_loss: 2.7902
Epoch 2/100
1/1 - 3s - 3s/step - accuracy: 0.0583 - loss: 2.7703 - val_accuracy: 0.0463 - val_loss: 2.7918
Epoch 3/100
1/1 - 3s - 3s/step - accuracy: 0.0667 - loss: 2.7771 - val_accuracy: 0.1530 - val_loss: 2.7855
Epoch 4/100
1/1 - 3s - 3s/step - accuracy: 0.0542 - loss: 2.7666 - val_accuracy: 0.0431 - val_loss: 2.7804
Epoch 5/100
1/1 - 3s - 3s/step - accuracy: 0.0833 - loss: 2.7612 - val_accuracy: 0.0348 - val_loss: 2.7766
Epoch 6/100
1/1 - 3s - 3s/step - accuracy: 0.1042 - loss: 2.7477 - val_accuracy: 0.0484 - val_loss: 2.7735
Epoch 7/100
1/1 - 3s - 3s/step - accuracy: 0.0917 - loss: 2.7329 - val_accuracy: 0.0816 - val_loss: 2.7702
Epoch 8/100
1/1 - 3s - 3s/step - accuracy: 0.0958 - loss: 2.7135 - val_accuracy: 0.0887 - val_loss: 2.7601
Epoch 9/100
1/1 - 3s - 3s/step - accuracy: 0.1500 - loss: 2.6467 - val_accuracy: 0.1165 - val_loss: 2.7264
Epoch 10/100
1/1 - 3s - 3s/step - accuracy: 0.1